<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/Emulator_Aug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')


# New FD script for building larger dataset

In [ ]:
# ============================================================
# run_fd_1layer_catalog_with_colloc.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
#
# NEW OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/colloc_raw/<IC_KEY>_colloc.npz
#
# The collocation file stores isolated centered grid points plus local FD
# diagnostics at those points:
#   t_sec, fd_step, macro_index, ix, iy, x_m, y_m
#   eta, uc, vc, f
#   deta_dt_fd, duc_dt_fd, dvc_dt_fd
#   etax_fd, etay_fd
#   ucx_fd, ucy_fd
#   vcx_fd, vcy_fd
#   div_fd, zeta_fd
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
ROOT_OUT  = "/content/drive/MyDrive/AI_EMUL/klein_ckpt_1L_big50"
ROOT_COLLOC = "/content/drive/MyDrive/AI_EMUL/klein_ml_1L_big50/colloc_raw"

# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = set(range(0, nt + 1, 50))

# -------------------------
# Colab batch controls
# -------------------------
# Run only keys[BATCH_START:BATCH_END] in this Colab session.
# Set BATCH_END = None to run to the end.
BATCH_START = 20
BATCH_END   = 40

# Restart-safe behavior: if final snapshot exists, skip that IC.
SKIP_COMPLETED = True
FINAL_SNAPSHOT = f"klein_step_{nt:06d}.npz"

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Collocation config
# -------------------------
SAVE_COLLOC = True
COLLOC_EVERY_N_STEPS = 20
N_COLLOC_PER_TIME = 256
COLLOC_WEIGHT_MODE = "grad_eta"   # "uniform" or "grad_eta"
COLLOC_SEED = 42
SAVE_COLLOC_COORDS = True

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5 * dx, Lx - 0.5 * dx, nx)
y_c = np.linspace(0.5 * dy, Ly - 0.5 * dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)

phi_c = np.pi * ((Yc / Ly) - 0.5)
f_c = fp * np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx + 1)
y_v = np.linspace(0.0, Ly, ny + 1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)

phi_u = np.pi * ((Yu / Ly) - 0.5)
phi_v = np.pi * ((Yv / Ly) - 0.5)
f_u = fp * np.sin(phi_u)
f_v = fp * np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5 * (h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5 * (h[-2, :] + twist_reflect_x(h[-2, :]))

    # u odd
    u[0, :]  = 0.5 * (u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5 * (u[-2, :] - twist_reflect_x(u[-2, :]))

    # v even
    v[0, :]  = 0.5 * (v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5 * (v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5 * (u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5 * (v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5 * (
        np.pad(a, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(a, ((0, 0), (0, 1)), mode="wrap")
    )

def avg_y(a):
    return 0.5 * (
        np.pad(a, ((1, 0), (0, 0)), mode="edge")
        + np.pad(a, ((0, 1), (0, 0)), mode="edge")
    )

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0, 0), (1, 0)), mode="wrap")
    R = np.pad(phi, ((0, 0), (0, 1)), mode="wrap")
    return (R - L) / (2.0 * dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1, 0), (0, 0)), mode="edge")
    B = np.pad(phi, ((0, 1), (0, 0)), mode="edge")
    return (B - T) / (2.0 * dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Centered FD diagnostics on emulator variables
# Used only for saved collocation diagnostics, not solver evolution.
# -------------------------
def ddx_center(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)

def ddy_center_edge(a):
    ap = np.pad(a, ((1, 1), (0, 0)), mode="edge")
    return (ap[2:, :] - ap[:-2, :]) / (2.0 * dy)

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0, 0), (1, 1)), mode="wrap")
    u_xx = (ue[:, :-2] - 2 * ue[:, 1:-1] + ue[:, 2:]) / dx**2

    ue2 = np.pad(u, ((1, 1), (0, 0)), mode="edge")
    u_yy = (ue2[:-2, :] - 2 * ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0, 0), (1, 1)), mode="wrap")
    v_xx = (ve[:, :-2] - 2 * ve[:, 1:-1] + ve[:, 2:]) / dx**2

    ve2 = np.pad(v, ((1, 1), (0, 0)), mode="edge")
    v_yy = (ve2[:-2, :] - 2 * ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0, 0), (1, 1)), mode="wrap")
    h_xx = (he[:, :-2] - 2 * he[:, 1:-1] + he[:, 2:]) / dx**2

    he2 = np.pad(h, ((1, 1), (0, 0)), mode="edge")
    h_yy = (he2[:-2, :] - 2 * he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u):
    return lap_u(lap_u(u))

def bih_v(v):
    return lap_v(lap_v(v))

def bih_c(h):
    return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0, 0), (1, 0)), mode="wrap")
    v_e = np.pad(v, ((0, 0), (0, 1)), mode="wrap")
    dv_dx = (v_e - v_w) / (2 * dx)

    u_s = np.pad(u, ((1, 0), (0, 0)), mode="edge")
    u_n = np.pad(u, ((0, 1), (0, 0)), mode="edge")
    du_dy = (u_n - u_s) / (2 * dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5 * (a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5 * (a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5 * (
        np.pad(s_c, ((0, 0), (1, 0)), mode="wrap")
        + np.pad(s_c, ((0, 0), (0, 1)), mode="wrap")
    )
    s_v = 0.5 * (
        np.pad(s_c, ((1, 0), (0, 0)), mode="edge")
        + np.pad(s_c, ((0, 1), (0, 0)), mode="edge")
    )

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5 * (uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u * lap_u(u) + nu4_u * bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v * lap_v(v) + nu4_v * bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h * lap_c(h) + nu4_h * bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5 * dt * k1u, v + 0.5 * dt * k1v, h + 0.5 * dt * k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5 * dt * k2u, v + 0.5 * dt * k2v, h + 0.5 * dt * k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt * k3u, v + dt * k3v, h + dt * k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt / 6.0) * (k1u + 2 * k2u + 2 * k3u + k4u)
    v_new = v + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)
    h_new = h + (dt / 6.0) * (k1h + 2 * k2h + 2 * k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Collocation helpers
# -------------------------
def init_colloc_store():
    return {
        "t_sec": [],
        "fd_step": [],
        "macro_index": [],
        "ix": [],
        "iy": [],
        "x_m": [],
        "y_m": [],
        "eta": [],
        "uc": [],
        "vc": [],
        "f": [],
        "deta_dt_fd": [],
        "duc_dt_fd": [],
        "dvc_dt_fd": [],
        "etax_fd": [],
        "etay_fd": [],
        "ucx_fd": [],
        "ucy_fd": [],
        "vcx_fd": [],
        "vcy_fd": [],
        "div_fd": [],
        "zeta_fd": [],
    }

def finalize_colloc_store(store):
    out = {}
    for k, v in store.items():
        if len(v) == 0:
            if k in ("fd_step", "macro_index", "ix", "iy"):
                out[k] = np.array([], dtype=np.int32)
            else:
                out[k] = np.array([], dtype=np.float32)
        else:
            out[k] = np.concatenate(v, axis=0)
    return out

def save_colloc_npz(ic_key, store):
    os.makedirs(ROOT_COLLOC, exist_ok=True)
    out_path = os.path.join(ROOT_COLLOC, f"{ic_key}_colloc.npz")

    save_dict = finalize_colloc_store(store)
    save_dict["ic_key"] = np.array(ic_key)
    save_dict["nx"] = np.int32(nx)
    save_dict["ny"] = np.int32(ny)
    save_dict["dx"] = np.float32(dx)
    save_dict["dy"] = np.float32(dy)
    save_dict["dt"] = np.float32(dt)
    save_dict["H"] = np.float32(H)
    save_dict["fp"] = np.float32(fp)

    np.savez_compressed(out_path, **save_dict)
    return out_path

def sample_collocation_indices(weight, npts, rng):
    flat_n = ny * nx
    npts = min(npts, flat_n)

    if weight is None:
        idx = rng.choice(flat_n, size=npts, replace=False)
    else:
        w = np.asarray(weight, dtype=np.float64).reshape(-1)
        w = np.maximum(w, 0.0)
        s = w.sum()
        if s <= 0.0:
            idx = rng.choice(flat_n, size=npts, replace=False)
        else:
            w = w / s
            idx = rng.choice(flat_n, size=npts, replace=False, p=w)

    iy = idx // nx
    ix = idx % nx
    return iy.astype(np.int32), ix.astype(np.int32)

def colloc_weight_from_eta(eta, mode="grad_eta"):
    if mode == "uniform":
        return None
    if mode == "grad_eta":
        ex = ddx_center(eta)
        ey = ddy_center_edge(eta)
        w = np.sqrt(ex * ex + ey * ey)
        return w + 1.0e-12
    return None

def get_centered_state_and_fd_diagnostics(u, v, h):
    """
    Return centered state, centered FD time tendencies, and local centered
    FD spatial diagnostics, all on the (ny, nx) centered grid.

    State:
        eta, uc, vc

    Time tendencies:
        deta_dt_fd, duc_dt_fd, dvc_dt_fd

    Spatial diagnostics:
        etax_fd, etay_fd
        ucx_fd, ucy_fd
        vcx_fd, vcy_fd
        div_fd, zeta_fd
    """
    u, v, h = apply_bc_1l(u, v, h)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    du, dv, dhdt = rhs_1l(u, v, h)

    deta_dt_fd = dhdt
    duc_dt_fd = center_from_u(du)
    dvc_dt_fd = center_from_v(dv)

    etax_fd = ddx_center(eta)
    etay_fd = ddy_center_edge(eta)

    ucx_fd = ddx_center(uc)
    ucy_fd = ddy_center_edge(uc)

    vcx_fd = ddx_center(vc)
    vcy_fd = ddy_center_edge(vc)

    div_fd = ucx_fd + vcy_fd
    zeta_fd = vcx_fd - ucy_fd

    return (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    )

def append_colloc_samples(store, step, macro_index, t, u, v, h, rng):
    (
        eta, uc, vc,
        deta_dt_fd, duc_dt_fd, dvc_dt_fd,
        etax_fd, etay_fd,
        ucx_fd, ucy_fd,
        vcx_fd, vcy_fd,
        div_fd, zeta_fd,
    ) = get_centered_state_and_fd_diagnostics(u, v, h)

    weight = colloc_weight_from_eta(eta, mode=COLLOC_WEIGHT_MODE)
    iy, ix = sample_collocation_indices(weight, N_COLLOC_PER_TIME, rng)

    store["t_sec"].append(np.full(len(iy), t, dtype=np.float32))
    store["fd_step"].append(np.full(len(iy), step, dtype=np.int32))
    store["macro_index"].append(np.full(len(iy), macro_index, dtype=np.int32))

    store["ix"].append(ix)
    store["iy"].append(iy)

    if SAVE_COLLOC_COORDS:
        store["x_m"].append(Xc[iy, ix].astype(np.float32))
        store["y_m"].append(Yc[iy, ix].astype(np.float32))
    else:
        store["x_m"].append(np.zeros(len(iy), dtype=np.float32))
        store["y_m"].append(np.zeros(len(iy), dtype=np.float32))

    store["eta"].append(eta[iy, ix].astype(np.float32))
    store["uc"].append(uc[iy, ix].astype(np.float32))
    store["vc"].append(vc[iy, ix].astype(np.float32))
    store["f"].append(f_c[iy, ix].astype(np.float32))

    store["deta_dt_fd"].append(deta_dt_fd[iy, ix].astype(np.float32))
    store["duc_dt_fd"].append(duc_dt_fd[iy, ix].astype(np.float32))
    store["dvc_dt_fd"].append(dvc_dt_fd[iy, ix].astype(np.float32))

    store["etax_fd"].append(etax_fd[iy, ix].astype(np.float32))
    store["etay_fd"].append(etay_fd[iy, ix].astype(np.float32))

    store["ucx_fd"].append(ucx_fd[iy, ix].astype(np.float32))
    store["ucy_fd"].append(ucy_fd[iy, ix].astype(np.float32))

    store["vcx_fd"].append(vcx_fd[iy, ix].astype(np.float32))
    store["vcy_fd"].append(vcy_fd[iy, ix].astype(np.float32))

    store["div_fd"].append(div_fd[iy, ix].astype(np.float32))
    store["zeta_fd"].append(zeta_fd[iy, ix].astype(np.float32))

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5 * h * (uc**2 + vc**2)
    pe = 0.5 * g * (eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5 * Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))

    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8, 4))
    plt.plot(tdays, (np.asarray(m_series) - M0) / M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series) - E0) / E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")

    # Restart-safe skip: do not rerun completed ICs.
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    final_path = os.path.join(ic_dir, FINAL_SNAPSHOT)
    if SKIP_COMPLETED and os.path.exists(final_path):
        print(f"[skip] {ic_key} already completed: {final_path}")
        return

    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    rng = np.random.default_rng(COLLOC_SEED)
    colloc_store = init_colloc_store()
    macro_index = 0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")
        macro_index += 1

    if SAVE_COLLOC and (0 % COLLOC_EVERY_N_STEPS == 0):
        append_colloc_samples(
            store=colloc_store,
            step=0,
            macro_index=macro_index,
            t=t,
            u=u, v=v, h=h,
            rng=rng,
        )

    for n in range(1, nt + 1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")
            macro_index += 1

        if SAVE_COLLOC and (n % COLLOC_EVERY_N_STEPS == 0):
            append_colloc_samples(
                store=colloc_store,
                step=n,
                macro_index=macro_index,
                t=t,
                u=u, v=v, h=h,
                rng=rng,
            )

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc * uc + vc * vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)

    if SAVE_COLLOC:
        colloc_path = save_colloc_npz(ic_key, colloc_store)
        print(f"[colloc] {ic_key} -> {colloc_path}")

    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    os.makedirs(ROOT_COLLOC, exist_ok=True)

    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)

    # Colab-friendly incremental batch selection.
    end = BATCH_END if BATCH_END is not None else len(keys)
    run_keys = keys[BATCH_START:end]

    print(f"Running batch keys[{BATCH_START}:{end}] -> {len(run_keys)} ICs")
    print("Batch ICs:", run_keys)
    print("Output root:", ROOT_OUT)
    print("Collocation root:", ROOT_COLLOC)
    print("Save steps:", sorted(SAVE_STEPS))

    for k in run_keys:
        run_ic(k)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz'

ICs in original bundle: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
  family gauss : ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05']
  family modon : ['test_modon_00', 'test_modon_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05']
  family wave  : ['wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']
  family rh    : ['test_rh_00', 'test_rh_01']
  family mix   : ['mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05']
[manifest] /content/drive/MyDrive/AI_EMUL/klein_ckpt_1L_big50/augmentation_manifest.csv
Running augmented batch specs[0:20] -> 20 ICs
Batch ICs: ['gauss_aug_000', 'gauss_aug_001', 'gauss_aug_002', 'gauss_aug_003', 'gauss_aug_004', 'g

# ML Emulator adjusted for large dataset

In [ ]:
# ============================================================
# train_rescnn_1layer_big50_augmented_resume.py
# ------------------------------------------------------------
# Configurable residual-CNN trainer for 1-layer SWE emulator
# on Klein beta-plane, with:
#   - multistep rollout data loss
#   - collocation state loss
#   - collocation tendency loss
#   - continuity residual (flux form)
#   - momentum residual (nonlinear)
#   - weak geostrophic-balance loss
#   - weak damping / smoothness loss
#
# Outputs go to:
#   /content/drive/MyDrive/AI_EMUL/training_runs/...
# ============================================================

import os
import sys
import glob
import csv
import time
import random
import re
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Import CollocBank
# ------------------------------------------------------------
sys.path.append("/content/drive/MyDrive/AI_4DVAR")
from colloc_bank import CollocBank

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# USER CONFIG
# ------------------------------------------------------------
class CFG:
    # -------- paths --------
    # Original code/helper location. CollocBank remains in AI_4DVAR.
    ROOT_CODE = "/content/drive/MyDrive/AI_4DVAR"

    # New expanded data and output root.
    ROOT_DATA = "/content/drive/MyDrive/AI_EMUL"
    ROOT_OUT  = "/content/drive/MyDrive/AI_EMUL"

    DATA_DIR_CANDIDATES = [
        f"{ROOT_DATA}/klein_ckpt_1L_big50",
    ]
    COLLOC_DIR = f"{ROOT_DATA}/klein_ml_1L_big50/colloc_raw"

    # New experiment tag.
    EXP_NAME = "rescnn_b12_c96_big50_aug_t6_p4_keweak"

    # -------- domain / physics --------
    NX = 256
    NY = 128
    Lx = 2.0e7
    Ly = 8.0e6
    H = 1000.0
    G = 9.81
    DT_MACRO = 50.0 * 30.0
    FMIN = 2.0e-5

    # -------- architecture --------
    N_BLOCKS = 12
    FEAT_CH  = 96
    HIDDEN   = 192

    # -------- optimization --------
    EPOCHS = 40
    BATCH_SIZE = 1
    LR = 1e-4
    WEIGHT_DECAY = 1e-6
    GRAD_CLIP = 1.0

    # -------- rollout --------
    # 8 x 50-step intervals gives a longer fine-time recursive training window.
    # If training is too slow in Colab, set ROLL_STEPS = 6.
    ROLL_STEPS = 8
    ROLL_GAMMA = 0.95

    # -------- data loading --------
    MAX_WINDOWS_PER_IC = 12
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

    # -------- collocation controls --------
    N_TIME_SLICES = 6
    PTS_PER_TIME  = 4

    # -------- loss weights --------
    LAMBDA_DATA       = 1.0
    LAMBDA_COLL_STATE = 0.05
    LAMBDA_COLL_TEND  = 0.1
    LAMBDA_CONT       = 0.2
    LAMBDA_MOM        = 0.5
    LAMBDA_GEO        = 0.01
    LAMBDA_SMOOTH     = 1e-4

    # -------- misc / restart --------
    SAVE_EVERY_EPOCH = 1
    PRINT_BATCH_EVERY = 25

    # Restart-safe Colab training. If AUTO_RESUME is True and
    # last_model.pt exists, training continues from it automatically.
    AUTO_RESUME = True
    RESUME_FROM = None

    # Optional cap for smoke tests. Use None for full training.
    MAX_BATCHES_PER_EPOCH = None

cfg = CFG()

cfg.DX = cfg.Lx / cfg.NX
cfg.DY = cfg.Ly / cfg.NY
cfg.CKPT_DIR = os.path.join(cfg.ROOT_OUT, "training_runs", cfg.EXP_NAME)
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def count_pairs_in_root(data_dir):
    if not os.path.exists(data_dir):
        return 0, 0
    ic_dirs = [d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)]
    n_ic = 0
    n_pairs = 0
    for ic_dir in ic_dirs:
        files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
        if len(files) >= 2:
            n_ic += 1
            n_pairs += (len(files) - 1)
    return n_ic, n_pairs

def autodetect_data_dir(candidates):
    print("[AutoDetect] checking candidate snapshot roots...")
    best_dir = None
    best_pairs = -1
    for d in candidates:
        n_ic, n_pairs = count_pairs_in_root(d)
        print(f"  {d}")
        print(f"     valid IC dirs = {n_ic}, total pairs = {n_pairs}")
        if n_pairs > best_pairs:
            best_pairs = n_pairs
            best_dir = d
    if best_dir is None or best_pairs <= 0:
        raise RuntimeError("No valid snapshot root found.")
    print(f"[AutoDetect] using: {best_dir}")
    return best_dir

_step_re = re.compile(r"klein_step_(\d+)\.npz")

def extract_step_from_path(path):
    m = _step_re.search(os.path.basename(path))
    if m is None:
        raise ValueError(f"Could not parse step from {path}")
    return int(m.group(1))

def save_checkpoint(path, model, optimizer, epoch, loss_history, data_dir):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss_history": loss_history,
            "data_dir": data_dir,
            "exp_name": cfg.EXP_NAME,
            "config": {k: v for k, v in cfg.__dict__.items() if not k.startswith("__")},
        },
        path,
    )

def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and "optimizer_state_dict" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    return ckpt.get("epoch", -1), ckpt.get("loss_history", []), ckpt

def save_loss_csv(path, loss_history):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "epoch", "train_total", "train_data",
            "train_coll_state", "train_coll_tend",
            "train_cont", "train_mom",
            "train_geo", "train_smooth"
        ])
        for row in loss_history:
            w.writerow(row)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class SWWindowDataset(Dataset):
    def __init__(self, data_dir, roll_steps=4, max_windows_per_ic=None):
        self.samples = []

        ic_dirs = sorted([d for d in glob.glob(os.path.join(data_dir, "*")) if os.path.isdir(d)])
        print(f"[Dataset] found {len(ic_dirs)} IC directories")

        for ic_dir in ic_dirs:
            ic_key = os.path.basename(ic_dir)
            files = sorted(glob.glob(os.path.join(ic_dir, "klein_step_*.npz")))
            print(f"[Dataset] {ic_key:20s} -> {len(files)} snapshot files")

            if len(files) < (roll_steps + 1):
                continue

            windows = []
            for i in range(len(files) - roll_steps):
                fseq = files[i:i + roll_steps + 1]
                macro_start_index = i + 1
                windows.append((fseq, ic_key, macro_start_index))

            if max_windows_per_ic is not None:
                windows = windows[:max_windows_per_ic]

            self.samples.extend(windows)

        if len(self.samples) == 0:
            raise RuntimeError("No usable training windows found.")

        print(f"[Dataset] total windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fseq, ic_key, macro_start_index = self.samples[idx]

        seq = []
        times = []
        steps = []

        for f in fseq:
            z = np.load(f)
            seq.append(np.stack([z["eta"], z["uc"], z["vc"]], axis=0).astype(np.float32))
            times.append(float(z["t"]))
            steps.append(int(extract_step_from_path(f)))

        seq = np.stack(seq, axis=0)

        return {
            "seq": torch.from_numpy(seq),
            "times": np.array(times, dtype=np.float32),
            "steps": np.array(steps, dtype=np.int32),
            "ic_key": ic_key,
            "macro_start_index": np.int32(macro_start_index),
        }

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
class ResBlock(nn.Module):
    def __init__(self, ch, dilation=1):
        super().__init__()
        pad = dilation
        self.c1 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=pad, dilation=dilation)
        self.act = nn.GELU()

    def forward(self, x):
        r = x
        x = self.act(self.c1(x))
        x = self.c2(x)
        return self.act(x + r)

class MultiStepContinuousResCNNModel(nn.Module):
    def __init__(self, in_ch=3, feat_ch=96, hidden=192, n_blocks=8):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
        )

        blocks = []
        dilations = [1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2]
        for i in range(n_blocks):
            blocks.append(ResBlock(feat_ch, dilation=dilations[i % len(dilations)]))
        self.resnet = nn.Sequential(*blocks)

        self.grid_head = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(feat_ch, 3, 3, padding=1),
        )

        self.query_mlp = nn.Sequential(
            nn.Linear(feat_ch + 3 + 3, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 3),
        )

        # zero-init safe start
        nn.init.zeros_(self.grid_head[-1].weight)
        nn.init.zeros_(self.grid_head[-1].bias)
        nn.init.zeros_(self.query_mlp[-1].weight)
        nn.init.zeros_(self.query_mlp[-1].bias)

    def encode(self, x):
        return self.resnet(self.stem(x))

    def grid_tendency(self, feat):
        return self.grid_head(feat)

    def _sample_local(self, field_1b, x_norm_detached, y_norm_detached):
        grid = torch.stack([x_norm_detached, y_norm_detached], dim=-1).view(1, -1, 1, 2)
        vals = F.grid_sample(
            field_1b, grid,
            mode="bilinear",
            padding_mode="border",
            align_corners=True,
        )
        return vals.squeeze(0).squeeze(-1).transpose(0, 1)

    def query_state(self, x_1b, feat_1b, x_norm, y_norm, tau):
        x_norm_s = x_norm.detach()
        y_norm_s = y_norm.detach()

        feat_local = self._sample_local(feat_1b, x_norm_s, y_norm_s)
        state0_local = self._sample_local(x_1b, x_norm_s, y_norm_s)

        q = torch.stack([x_norm, y_norm, tau], dim=-1)
        inp = torch.cat([feat_local, state0_local, q], dim=-1)

        delta = self.query_mlp(inp)
        return state0_local + tau.unsqueeze(-1) * delta

# ------------------------------------------------------------
# Collocation sampling
# ------------------------------------------------------------
def sample_multitime_colloc(colloc_bank, ic_key, macro_index, n_time_slices=4, pts_per_time=4):
    n_request = max(n_time_slices * pts_per_time * 8, 64)
    base = colloc_bank.sample_nearest_macro(
        ic_key=ic_key,
        macro_index=macro_index,
        npts=n_request,
        replace=True,
    )

    t_sec = np.asarray(base["t_sec"]).reshape(-1)
    if t_sec.size == 0:
        return base

    order = np.argsort(t_sec)
    bins = np.array_split(order, n_time_slices)

    chosen = []
    for b in bins:
        if len(b) == 0:
            continue
        take = min(pts_per_time, len(b))
        idx = np.random.choice(b, size=take, replace=False)
        chosen.append(idx)

    if len(chosen) == 0:
        chosen = [order[:min(pts_per_time, len(order))]]

    chosen = np.concatenate(chosen, axis=0)

    out = {}
    for k, arr in base.items():
        arr = np.asarray(arr)
        if arr.ndim >= 1 and arr.shape[0] == len(t_sec):
            out[k] = arr[chosen]
        else:
            out[k] = arr
    return out

# ------------------------------------------------------------
# Derivative / balance helpers
# ------------------------------------------------------------
def safe_coriolis_torch(f, fmin):
    sign = torch.sign(f)
    sign = torch.where(sign == 0.0, torch.ones_like(sign), sign)
    return sign * torch.clamp(torch.abs(f), min=fmin)

# ------------------------------------------------------------
# Collocation losses
# ------------------------------------------------------------
def continuous_interval_losses_multitime(model, x_1b, feat_1b, colloc, t_start, dt_macro):
    x_m = torch.as_tensor(colloc["x_m"], dtype=torch.float32, device=device)
    y_m = torch.as_tensor(colloc["y_m"], dtype=torch.float32, device=device)
    t_sec = torch.as_tensor(colloc["t_sec"], dtype=torch.float32, device=device)

    tau = (t_sec - t_start) / dt_macro
    tau = torch.clamp(tau, 0.0, 1.0)

    x_norm = (2.0 * x_m / cfg.Lx) - 1.0
    y_norm = (2.0 * y_m / cfg.Ly) - 1.0

    x_norm = x_norm.clone().detach().requires_grad_(True)
    y_norm = y_norm.clone().detach().requires_grad_(True)
    tau    = tau.clone().detach().requires_grad_(True)

    state_hat = model.query_state(x_1b, feat_1b, x_norm, y_norm, tau)
    eta_hat = state_hat[:, 0]
    u_hat   = state_hat[:, 1]
    v_hat   = state_hat[:, 2]
    h_hat   = cfg.H + eta_hat

    def grads_of(field):
        gx, gy, gt = torch.autograd.grad(
            field.sum(), [x_norm, y_norm, tau],
            create_graph=True, retain_graph=True
        )
        return gx, gy, gt

    eta_xn, eta_yn, eta_tau = grads_of(eta_hat)
    u_xn,   u_yn,   u_tau   = grads_of(u_hat)
    v_xn,   v_yn,   v_tau   = grads_of(v_hat)

    eta_x = eta_xn * (2.0 / cfg.Lx)
    eta_y = eta_yn * (2.0 / cfg.Ly)
    h_x = eta_x
    h_y = eta_y

    u_x = u_xn * (2.0 / cfg.Lx)
    u_y = u_yn * (2.0 / cfg.Ly)
    v_x = v_xn * (2.0 / cfg.Lx)
    v_y = v_yn * (2.0 / cfg.Ly)

    eta_t = eta_tau / dt_macro
    h_t = eta_t
    u_t = u_tau / dt_macro
    v_t = v_tau / dt_macro

    hu = h_hat * u_hat
    hv = h_hat * v_hat
    hu_xn, _, _ = grads_of(hu)
    _, hv_yn, _ = grads_of(hv)
    hu_x = hu_xn * (2.0 / cfg.Lx)
    hv_y = hv_yn * (2.0 / cfg.Ly)

    eta_true = torch.as_tensor(colloc["eta"], dtype=torch.float32, device=device)
    u_true   = torch.as_tensor(colloc["uc"],  dtype=torch.float32, device=device)
    v_true   = torch.as_tensor(colloc["vc"],  dtype=torch.float32, device=device)

    deta_dt_fd = torch.as_tensor(colloc["deta_dt_fd"], dtype=torch.float32, device=device)
    duc_dt_fd  = torch.as_tensor(colloc["duc_dt_fd"],  dtype=torch.float32, device=device)
    dvc_dt_fd  = torch.as_tensor(colloc["dvc_dt_fd"],  dtype=torch.float32, device=device)

    f_fd = torch.as_tensor(colloc["f"], dtype=torch.float32, device=device)

    # state collocation
    loss_coll_state = ((eta_hat - eta_true) ** 2).mean()
    loss_coll_state += ((u_hat - u_true) ** 2).mean()
    loss_coll_state += ((v_hat - v_true) ** 2).mean()

    # tendency collocation
    loss_coll_tend = ((eta_t - deta_dt_fd) ** 2).mean()
    loss_coll_tend += ((u_t - duc_dt_fd) ** 2).mean()
    loss_coll_tend += ((v_t - dvc_dt_fd) ** 2).mean()

    # continuity in flux form
    resid_cont = h_t + hu_x + hv_y
    loss_cont = (resid_cont ** 2).mean()

    # nonlinear momentum
    adv_u = u_hat * u_x + v_hat * u_y
    adv_v = u_hat * v_x + v_hat * v_y
    resid_u = u_t + adv_u - f_fd * v_hat + cfg.G * h_x
    resid_v = v_t + adv_v + f_fd * u_hat + cfg.G * h_y
    loss_mom = (resid_u ** 2).mean() + (resid_v ** 2).mean()

    # weak geostrophic balance penalty
    f_safe = safe_coriolis_torch(f_fd, cfg.FMIN)
    u_g = -(cfg.G / f_safe) * eta_y
    v_g =  (cfg.G / f_safe) * eta_x
    loss_geo = ((u_hat - u_g) ** 2).mean() + ((v_hat - v_g) ** 2).mean()

    # weak smoothness / damping penalty
    loss_smooth = (u_x ** 2 + u_y ** 2 + v_x ** 2 + v_y ** 2).mean()

    return loss_coll_state, loss_coll_tend, loss_cont, loss_mom, loss_geo, loss_smooth

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
def train():
    data_dir = autodetect_data_dir(cfg.DATA_DIR_CANDIDATES)

    if not os.path.exists(cfg.COLLOC_DIR):
        raise RuntimeError(f"COLLOC_DIR does not exist: {cfg.COLLOC_DIR}")

    colloc_bank = CollocBank(cfg.COLLOC_DIR, verbose=True)

    dataset = SWWindowDataset(
        data_dir,
        roll_steps=cfg.ROLL_STEPS,
        max_windows_per_ic=cfg.MAX_WINDOWS_PER_IC,
    )

    loader = DataLoader(
        dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        drop_last=False,
    )

    model = MultiStepContinuousResCNNModel(
        feat_ch=cfg.FEAT_CH,
        hidden=cfg.HIDDEN,
        n_blocks=cfg.N_BLOCKS,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    start_epoch = 0
    loss_history = []

    # --------------------------------------------------------
    # Restart / resume logic
    # --------------------------------------------------------
    resume_path = cfg.RESUME_FROM
    auto_last = os.path.join(cfg.CKPT_DIR, "last_model.pt")

    if resume_path is None and cfg.AUTO_RESUME and os.path.exists(auto_last):
        resume_path = auto_last

    if resume_path is not None and os.path.exists(resume_path):
        print(f"[Resume] loading checkpoint: {resume_path}")
        loaded_epoch, loaded_history, _ = load_checkpoint(resume_path, model, optimizer)
        start_epoch = loaded_epoch + 1
        loss_history = loaded_history
        print(f"[Resume] continuing from epoch {start_epoch}")
    else:
        print("[Resume] no checkpoint found; starting from scratch")

    best_total = min([row[1] for row in loss_history], default=float("inf"))

    if start_epoch >= cfg.EPOCHS:
        print(f"[Train] checkpoint already reached epoch {start_epoch-1}; EPOCHS={cfg.EPOCHS}. Nothing to do.")
        return

    print(f"[Train] output dir            = {cfg.CKPT_DIR}")
    print(f"[Train] windows               = {len(dataset)}")
    print(f"[Train] batches/epoch         = {len(loader)}")
    print(f"[Train] N_BLOCKS              = {cfg.N_BLOCKS}")
    print(f"[Train] FEAT_CH               = {cfg.FEAT_CH}")
    print(f"[Train] LR                    = {cfg.LR}")
    print(f"[Train] N_TIME_SLICES         = {cfg.N_TIME_SLICES}")
    print(f"[Train] PTS_PER_TIME          = {cfg.PTS_PER_TIME}")
    print(f"[Train] colloc/interval       = {cfg.N_TIME_SLICES * cfg.PTS_PER_TIME}")
    print(f"[Train] DT_MACRO              = {cfg.DT_MACRO} s")
    print(f"[Train] ROLL_STEPS            = {cfg.ROLL_STEPS}")
    print(f"[Train] MAX_WINDOWS_PER_IC    = {cfg.MAX_WINDOWS_PER_IC}")
    print(f"[Train] AUTO_RESUME           = {cfg.AUTO_RESUME}")

    for epoch in range(start_epoch, cfg.EPOCHS):
        t0 = time.time()
        model.train()

        run_total = 0.0
        run_data = 0.0
        run_cstate = 0.0
        run_ctend = 0.0
        run_cont = 0.0
        run_mom = 0.0
        run_geo = 0.0
        run_smooth = 0.0

        for ib, batch in enumerate(loader):
            if cfg.MAX_BATCHES_PER_EPOCH is not None and ib >= cfg.MAX_BATCHES_PER_EPOCH:
                print(f"[debug] stopping epoch early at batch {ib}")
                break

            seq = batch["seq"].to(device, non_blocking=True)
            times = batch["times"].numpy()
            ic_keys = batch["ic_key"]
            macro_start = batch["macro_start_index"].numpy()

            B = seq.shape[0]
            optimizer.zero_grad(set_to_none=True)

            loss_data = 0.0
            loss_cstate = 0.0
            loss_ctend = 0.0
            loss_cont = 0.0
            loss_mom = 0.0
            loss_geo = 0.0
            loss_smooth = 0.0
            n_phys = 0

            x = seq[:, 0]

            for k in range(cfg.ROLL_STEPS):
                feat = model.encode(x)
                xdot_grid = model.grid_tendency(feat)
                x_next = x + cfg.DT_MACRO * xdot_grid

                truth_next = seq[:, k + 1]
                loss_data = loss_data + (cfg.ROLL_GAMMA ** k) * ((x_next - truth_next) ** 2).mean()

                for b in range(B):
                    ic_key = ic_keys[b]
                    macro_idx = int(macro_start[b] + k)
                    t_start = float(times[b, k])

                    if not colloc_bank.has_ic(ic_key):
                        continue

                    colloc = sample_multitime_colloc(
                        colloc_bank=colloc_bank,
                        ic_key=ic_key,
                        macro_index=macro_idx,
                        n_time_slices=cfg.N_TIME_SLICES,
                        pts_per_time=cfg.PTS_PER_TIME,
                    )

                    l_cstate, l_ctend, l_cont, l_mom, l_geo, l_smooth = continuous_interval_losses_multitime(
                        model=model,
                        x_1b=x[b:b+1],
                        feat_1b=feat[b:b+1],
                        colloc=colloc,
                        t_start=t_start,
                        dt_macro=cfg.DT_MACRO,
                    )

                    loss_cstate += l_cstate
                    loss_ctend += l_ctend
                    loss_cont += l_cont
                    loss_mom += l_mom
                    loss_geo += l_geo
                    loss_smooth += l_smooth
                    n_phys += 1

                x = x_next

            if n_phys > 0:
                loss_cstate = loss_cstate / n_phys
                loss_ctend = loss_ctend / n_phys
                loss_cont = loss_cont / n_phys
                loss_mom = loss_mom / n_phys
                loss_geo = loss_geo / n_phys
                loss_smooth = loss_smooth / n_phys
            else:
                zero = torch.tensor(0.0, device=device)
                loss_cstate = zero
                loss_ctend = zero
                loss_cont = zero
                loss_mom = zero
                loss_geo = zero
                loss_smooth = zero

            loss = (
                cfg.LAMBDA_DATA       * loss_data
                + cfg.LAMBDA_COLL_STATE * loss_cstate
                + cfg.LAMBDA_COLL_TEND  * loss_ctend
                + cfg.LAMBDA_CONT       * loss_cont
                + cfg.LAMBDA_MOM        * loss_mom
                + cfg.LAMBDA_GEO        * loss_geo
                + cfg.LAMBDA_SMOOTH     * loss_smooth
            )

            if not torch.isfinite(loss):
                print("Non-finite diagnostics:")
                print("  data   =", float(loss_data.detach().cpu()))
                print("  cstate =", float(loss_cstate.detach().cpu()))
                print("  ctend  =", float(loss_ctend.detach().cpu()))
                print("  cont   =", float(loss_cont.detach().cpu()))
                print("  mom    =", float(loss_mom.detach().cpu()))
                print("  geo    =", float(loss_geo.detach().cpu()))
                print("  smooth =", float(loss_smooth.detach().cpu()))
                raise RuntimeError(f"Non-finite loss at epoch={epoch}, batch={ib}: {loss.item()}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            optimizer.step()

            run_total += float(loss.detach().cpu())
            run_data += float(loss_data.detach().cpu())
            run_cstate += float(loss_cstate.detach().cpu())
            run_ctend += float(loss_ctend.detach().cpu())
            run_cont += float(loss_cont.detach().cpu())
            run_mom += float(loss_mom.detach().cpu())
            run_geo += float(loss_geo.detach().cpu())
            run_smooth += float(loss_smooth.detach().cpu())

            if (ib % cfg.PRINT_BATCH_EVERY) == 0:
                print(
                    f"[epoch {epoch:03d} | batch {ib:04d}/{len(loader):04d}] "
                    f"total={float(loss.detach().cpu()):.6e} "
                    f"data={float(loss_data.detach().cpu()):.6e} "
                    f"cstate={float(loss_cstate.detach().cpu()):.6e} "
                    f"ctend={float(loss_ctend.detach().cpu()):.6e} "
                    f"cont={float(loss_cont.detach().cpu()):.6e} "
                    f"mom={float(loss_mom.detach().cpu()):.6e} "
                    f"geo={float(loss_geo.detach().cpu()):.6e} "
                    f"smooth={float(loss_smooth.detach().cpu()):.6e}"
                )

        n_batches = max(
            min(len(loader), cfg.MAX_BATCHES_PER_EPOCH)
            if cfg.MAX_BATCHES_PER_EPOCH is not None else len(loader),
            1,
        )
        ep_total = run_total / n_batches
        ep_data = run_data / n_batches
        ep_cstate = run_cstate / n_batches
        ep_ctend = run_ctend / n_batches
        ep_cont = run_cont / n_batches
        ep_mom = run_mom / n_batches
        ep_geo = run_geo / n_batches
        ep_smooth = run_smooth / n_batches

        loss_history.append([
            epoch, ep_total, ep_data, ep_cstate, ep_ctend, ep_cont, ep_mom, ep_geo, ep_smooth
        ])

        print(
            f"Epoch {epoch:03d} done | "
            f"total={ep_total:.6e} | data={ep_data:.6e} | "
            f"cstate={ep_cstate:.6e} | ctend={ep_ctend:.6e} | "
            f"cont={ep_cont:.6e} | mom={ep_mom:.6e} | "
            f"geo={ep_geo:.6e} | smooth={ep_smooth:.6e} | "
            f"time={time.time() - t0:.1f}s"
        )

        last_ckpt = os.path.join(cfg.CKPT_DIR, "last_model.pt")
        save_checkpoint(last_ckpt, model, optimizer, epoch, loss_history, data_dir)

        if ((epoch + 1) % cfg.SAVE_EVERY_EPOCH) == 0:
            save_checkpoint(
                os.path.join(cfg.CKPT_DIR, f"model_epoch_{epoch:03d}.pt"),
                model, optimizer, epoch, loss_history, data_dir
            )

        save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

        if ep_total < best_total:
            best_total = ep_total
            best_ckpt = os.path.join(cfg.CKPT_DIR, "best_model.pt")
            save_checkpoint(best_ckpt, model, optimizer, epoch, loss_history, data_dir)
            print(f"[Best] epoch={epoch:03d} train_total={ep_total:.6e} -> {best_ckpt}")

    final_ckpt = os.path.join(cfg.CKPT_DIR, "final_model.pt")
    save_checkpoint(final_ckpt, model, optimizer, cfg.EPOCHS - 1, loss_history, data_dir)
    save_loss_csv(os.path.join(cfg.CKPT_DIR, "loss_history.csv"), loss_history)

    print("[Train] finished.")
    print(f"[Train] final checkpoint: {final_ckpt}")

if __name__ == "__main__":
    train()